# Qwen3-4B - v9 fine-tune on Colab T4

**Hardware:** Tesla T4 (sm_75, 15.6 GB VRAM)

**T4 is Turing - it has NO native BF16 tensor cores and NO FlashAttention 2.**
This notebook therefore uses 4-bit NF4 + FP16 compute + SDPA.

Edit **Cell 3 (CONFIG)** to change anything. Then Runtime > Run all.

In [ ]:
import subprocess, sys, importlib

!pip -q install --no-cache-dir \
  "transformers==4.57.6" "peft==0.21.1" "datasets==4.8.5" \
  "accelerate==1.15.0" "sentencepiece" "protobuf"

# --- why these two are removed on this runtime -------------------------
# bitsandbytes 0.45.5 : libbitsandbytes_cpu.so -> undefined symbol
#                       cquantize_blockwise_fp32_nf4  (Python 3.13 build)
# torchao 0.10.0      : peft 0.21.1 requires >0.16.0 and RAISES on import
# This notebook therefore trains in fp16 + LoRA, never 4-bit NF4.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                "bitsandbytes", "torchao"], check=False)

import torch, transformers, peft, accelerate
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("peft        :", peft.__version__)
print("accelerate  :", accelerate.__version__)
for m in ("bitsandbytes", "torchao"):
    sys.modules.pop(m, None)
    try:
        importlib.import_module(m)
        print(m, ": STILL PRESENT (unexpected)")
    except Exception:
        print(m, ": removed OK")

assert torch.cuda.is_available(), "no GPU - set Runtime > Change runtime type > T4"
p = torch.cuda.get_device_properties(0)
print("GPU         :", torch.cuda.get_device_properties(0).name)
print("VRAM        : %.2f GB" % (p.total_memory/1e9))
print("capability  : %d.%d" % (p.major, p.minor))


## CONFIG - edit here

In [ ]:
import torch

CONFIG = {
    # ---- BASE MODEL ----
    # Option 1 (RECOMMENDED): clean base. Breaks the v7h/v8 stacking chain and
    # removes any inherited decline-to-act bias, giving an honest baseline.
    "base_model": "Qwen/Qwen3-4B",
    # Option 2: continue the chain from v7h.
    # "base_model": "unsloth/Qwen3-4B",

    # ---- DATA (loaded from Drive, Cell 4) ----
    # Order matters: tool-calling first, refusal reduction second,
    # prompt-injection resistance third.
    "datasets": [
        "train_v8_injection.jsonl",   # 600 rows - injection vs tool-use
        "train_v6.jsonl",             # 890 rows - general assistant
        "compliance_clean.jsonl",     # 514 rows - refusal reduction
    ],
    "val_file": "val_v6.jsonl",
    "max_samples": None,

    # ---- QLoRA: matches v8 lineage (r=16, alpha=16, dropout 0.0) ----
    "lora_r": 16,
    "lora_alpha": 16,
    "lora_dropout": 0.0,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],

    # ---- T4-SAFE TRAINING ----
    "max_seq_length": 1024,     # lower this first if you OOM: 768
    "per_device_batch": 1,
    "grad_accum": 16,           # effective batch 16
    "learning_rate": 2e-4,
    "epochs": 1,
    "max_steps": -1,            # -1 = let epochs drive it
    "warmup_ratio": 0.03,
    "weight_decay": 0.0,
    "lr_scheduler": "cosine",
    "logging_steps": 25,
    "save_steps": 100,
    "seed": 42,
    "output_dir": "/content/qwen3-4b-v9",

    # ---- T4 HARDWARE FORCED VALUES (do not change) ----
    "compute_dtype": torch.float16,   # NOT bfloat16 - T4 has no BF16 cores
    "attn_implementation": "sdpa",   # NOT flash_attention_2
}
for k, v in CONFIG.items():
    print(" ", k, "=", v)

## Cell 4 - load data from Drive

Put the `.jsonl` files in your Drive and set `DATA_DIR` below to that folder.

In [ ]:
import os, json, random
from google.colab import drive

drive.mount("/content/drive")

NEEDED = CONFIG["datasets"] + [CONFIG["val_file"]]
ROOT = "/content/drive/MyDrive"

def find_data_dir(root, needed):
    """Return the first directory under root that holds the required .jsonl files."""
    if all(os.path.exists(os.path.join(root, n)) for n in needed):
        return root
    for dirpath, dirnames, filenames in os.walk(root):
        if all(n in filenames for n in needed):
            return dirpath
    return None

DATA_DIR = find_data_dir(ROOT, NEEDED)

if DATA_DIR is None:
    raise SystemExit(
        f"""Could not find all of these under {ROOT}:

    {NEEDED}

Searched MyDrive root and every subfolder. Either put the files in Drive,
or hardcode DATA_DIR = "/path/to/folder" directly below."""
    )

print("DATA_DIR =", DATA_DIR)
print()
for f in sorted(os.listdir(DATA_DIR)):
    if f.endswith(".jsonl"):
        print("  %9.2f MB  %s" % (os.path.getsize(os.path.join(DATA_DIR, f))/1e6, f))

In [ ]:
from datasets import Dataset

def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    return rows

parts = []
for fn in CONFIG["datasets"]:
    rows = load_jsonl(os.path.join(DATA_DIR, fn))
    print("  %-32s %6d rows" % (fn, len(rows)))
    parts += rows

random.Random(CONFIG["seed"]).shuffle(parts)
if CONFIG["max_samples"]:
    parts = parts[: CONFIG["max_samples"]]
print("\nTOTAL TRAIN: %d rows" % len(parts))

train_ds = Dataset.from_list(parts)
print(train_ds)

val_ds = None
vp = os.path.join(DATA_DIR, CONFIG["val_file"])
if os.path.exists(vp):
    val_ds = Dataset.from_list(load_jsonl(vp))
    print("VAL: %d rows" % len(val_ds))

## Cell 6 - load model in 4-bit

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model

# TrainingArguments is imported HERE on purpose: the Trainer cell below uses it
# but does not import it itself. Dropping it causes a NameError one cell later.
#
# fp16, not 4-bit. bitsandbytes' compiled library is unusable on this runtime
# (undefined symbol: cquantize_blockwise_fp32_nf4), so QLoRA NF4 is not an
# option. Qwen3-4B fp16 is ~8.9 GB and fits the 16 GB T4 comfortably.
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], use_fast=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["base_model"],
    dtype=torch.float16,
    device_map={"": 0},
    attn_implementation=CONFIG["attn_implementation"],
    low_cpu_mem_usage=True,
)
model.enable_input_require_grads()    # required with gradient checkpointing
model.config.use_cache = False        # required with gradient checkpointing

print("loaded:", model.config._name_or_path)
print("dtype :", next(model.parameters()).dtype)
print("VRAM  : %.2f GB" % (torch.cuda.memory_allocated()/1e9))

lora_cfg = LoraConfig(
    r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=CONFIG["target_modules"],
    bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
free, total = torch.cuda.mem_get_info()
print("VRAM w/ LoRA: %.2f GB" % (torch.cuda.memory_allocated()/1e9))
print("free/total  : %.2f / %.2f GB" % (free/1e9, total/1e9))


## Cell 7 - train

FP16 + GradScaler. If OOM, drop `max_seq_length` to 768 - cheapest lever.

In [ ]:
import glob, os
from transformers import Trainer   # plain Trainer: this cell does its own tokenizing

def _tools_for(batch, i):
    """Return the tool schema for row i, or None when the row has none."""
    tl = batch.get("tools")
    if not isinstance(tl, list):
        return None
    return tl[i] if tl[i] else None

def tok_fn(batch):
    # tools= is passed through: the 600 tool-calling rows carry their schemas
    # in a "tools" column, and without this the chat template strips them.
    has_tools = "tools" in batch
    texts = [
        tok.apply_chat_template(
            m,
            tools=_tools_for(batch, i) if has_tools else None,
            tokenize=False,
            add_generation_prompt=False,
        )
        for i, m in enumerate(batch["messages"])
    ]
    enc = tok(texts, truncation=True,
              max_length=CONFIG["max_seq_length"], padding="max_length")
    enc["labels"] = list(enc["input_ids"])
    return enc

train_tok = train_ds.map(tok_fn, batched=True, remove_columns=train_ds.column_names)
val_tok = None
if val_ds is not None:
    val_tok = val_ds.map(tok_fn, batched=True, remove_columns=val_ds.column_names)

args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    per_device_train_batch_size=CONFIG["per_device_batch"],
    gradient_accumulation_steps=CONFIG["grad_accum"],
    num_train_epochs=CONFIG["epochs"],
    max_steps=CONFIG["max_steps"],
    learning_rate=CONFIG["learning_rate"],
    lr_scheduler_type=CONFIG["lr_scheduler"],
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    logging_steps=CONFIG["logging_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=3,
    fp16=True,                 # GradScaler - T4 safe
    bf16=False,                # T4 has no BF16 tensor cores
    gradient_checkpointing=True,
    optim="adamw_torch",       # NOT paged_adamw_8bit: needs bitsandbytes
    report_to=[],
    seed=CONFIG["seed"],
    group_by_length=True,
    remove_unused_columns=False,
)

# Resume across Colab sessions: the free tier caps a session at ~3h20m, so a
# run that outlasts it must continue from the newest checkpoint.
RESUME = None
_ckpts = sorted(glob.glob(os.path.join(CONFIG["output_dir"], "checkpoint-*")),
                key=lambda p: int(p.rsplit("-", 1)[1]))
if _ckpts:
    RESUME = _ckpts[-1]
    print("resuming from:", RESUME)
else:
    print("no checkpoint found - starting fresh")

trainer = Trainer(model=model, args=args,
                  train_dataset=train_tok, eval_dataset=val_tok)
trainer.train(resume_from_checkpoint=RESUME)

OUT = os.path.join(CONFIG["output_dir"], "adapter")
os.makedirs(OUT, exist_ok=True)
trainer.model.save_pretrained(OUT)
tok.save_pretrained(OUT)
print("adapter saved ->", OUT)


## Cell 8 - report + save to Drive

In [ ]:
hist = [h for h in trainer.state.log_history if "loss" in h]
if hist:
    print("initial loss:", hist[0]["loss"])
    print("final   loss:", hist[-1]["loss"])
print("peak VRAM: %.2f GB" % (torch.cuda.max_memory_allocated()/1e9))
print("total steps:", trainer.state.global_step)

try:
    import shutil
    DEST = "/content/drive/MyDrive/model-training/output/full4b_v9"
    os.makedirs(os.path.dirname(DEST), exist_ok=True)
    shutil.rmtree(DEST, ignore_errors=True)
    shutil.copytree(OUT, DEST)
    print("COPIED TO DRIVE ->", DEST)
except Exception as e:
    print("Drive copy failed:", e)

## Cell 9 - merge + GGUF (optional, inert)

Uncomment to produce `merged-v9` in bf16/fp16 for GGUF conversion.
Quantization should run on your laptop with llama.cpp (**inference/quantize only - never train there**).

In [ ]:
# from transformers import AutoModelForCausalLM as M
# from peft import PeftModel
# import gc
# del trainer, model
# gc.collect(); torch.cuda.empty_cache()
# merged = M.from_pretrained(CONFIG["base_model"], dtype=torch.float16, device_map={"":0})
# merged = PeftModel.from_pretrained(merged, OUT).merge_and_unload()
# merged.save_pretrained("/content/qwen3-4b-v9/merged", safe_serialization=True)
# tok.save_pretrained("/content/qwen3-4b-v9/merged")
# print("merged -> /content/qwen3-4b-v9/merged")
#
# On a machine with llama.cpp:
#   python convert_hf_to_gguf.py /content/qwen3-4b-v9/merged --outfile merged-v9-f16.gguf --outtype f16
#   ./llama-quantize merged-v9-f16.gguf merged-v9-q4_k_m.gguf Q4_K_M

print("merge cell - intentionally inert")

---
## T4 gotchas

| Symptom | Cause | Fix |
|---|---|---|
| `FlashAttention only supports Ampere+` | T4 is Turing sm_75 | `attn_implementation="sdpa"` (set) |
| loss becomes `nan` | BF16 on T4 | `fp16=True, bf16=False` (set) |
| CUDA out of memory | sequence too long | `max_seq_length` 1024 -> 768 |
| session dies mid-run | free tier, no background exec | keep this tab pinned |

### Execution limit
Your Google AI **Pro** tier grants *priority access to faster accelerators*.
**Uninterrupted background execution is Ultra-only**, so a free-tier session is
killed if this tab goes idle. v8 took 74.5 min on a much larger laptop GPU; a
clean-base run here may take **several hours**. Keep the tab open and active.